In [1]:
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field
import uuid

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, END, START, MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

c:\Users\Ashutosh Pandey\Desktop\Machine Learning\LangGraph-In-Depth\LangGraph\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
store = InMemoryStore()

In [3]:
# for creating memory also we need llm
Extractor_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

In [4]:
#memory deciding factor pydantic
class MemoryDecision(BaseModel):
    should_store: bool = Field(description="wheather to store the memory")
    memories: list[str] = Field(default_factory=list,description="list of memories to store")

In [5]:
memory_extractor = Extractor_llm.with_structured_output(MemoryDecision)

In [7]:
# creating the chat node function
def chatnode(state: MessagesState, config: RunnableConfig, store: BaseStore):
    # taking the thread id:
    user_id = config["configurable"]["user_id"]
    
    # creating the namespace:
    namespace = ("user", user_id, "details")
    
    #getting the last message from the response
    #here we will be sending two things system message and the alst message
    last_message = state["messages"][-1].content
    
    #this memory decision is the object of our memory decision pydantic model.
    decision: MemoryDecision = memory_extractor.invoke([
        SystemMessage(
            content= (
                "extract the long term memory from the user messgae. \n"
                "only store stable and user-specific info (identity, preferences, ongoing projects) \n"
                "Do not store transient info \n"
                "return should_write = fasle if nothing is worth storing. \n"
                "each memory should be a short atomic sentence.")
        ),
        # so we are giving the system message plus the users last message here.
        {"role": "user", "content": last_message},
    ])
    
    #now we will check if the decision is to store or not
    if decision.should_store:
        for memory in decision.memories:
            store.put(namespace, str(uuid.uuid4()), {"data": memory})
            
            
    # also note that we are not using memory , not even responding with the LLM       
    # now we will hard code the messgae that we have noted to put this in the memory
    return {"messages": {{"role": "assistant", "content": "i have noted this down!!"}}}
    